In [ ]:
from unittest import result
from deepagents import DeepAgentState, create_deep_agent
from numpy import str_
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Any, Dict
from parser import bin_to_dataframe_optimized
from langchain_core.tools import tool
import pandas as pd 
import numpy as np
import traceback
import message_information
from pandas import DataFrame
from typing_extensions import Annotated, TypedDict
from langgraph.prebuilt import InjectedState, ToolNode
from langgraph.types import Command
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool, InjectedToolCallId
import asyncio

load_dotenv()


class AgentState(DeepAgentState):
    result_df: DataFrame
    message_dfs: Dict 
    

class StructuredPandasQueryOutput(BaseModel):
    pandas_code: str = Field(..., description="The Pandas code generated by the agent")

    def to_dict(self) -> Dict[str, Any]:
        return {"pandas_code": self.pandas_code}


system_prompt = """ 
You are an expert in analyzing ArduPilot Logs. You will be given access to pre-processed telemetry data that has already been extracted from a .bin file.  
The data is stored in memory as a dictionary of Pandas DataFrames, keyed by message type (e.g., GPS, ATT, VIBE, CTUN).  
Users will ask you questions about the flight, and you will use the stored DataFrames to answer the questions.  

Your role is to analyze the user question and generate meaningful insights/reports/analytics from the data.

You can do the following:
1) Doing normal conversation with user
- Do greeting
- Do farewell
- Don't do small talks
- Answer if user is asking about your capabilities/potential.
- Do not answer questions unrelated to the overall goal of being a chatbot that can analyze ArduPilot Logs. Instead, tell the user that you are a chatbot that can analyze ArduPilot Logs and can't answer general questions. 
- If the user’s request is ambiguous, proactively ask clarifying questions before proceeding with analysis.

2) Analyzing the log file
- Analyze the log file to answer the user question.

Here is a simple general process that you can follow to answer the user question:
Step 0:(Optional) Ask clarifications
- Sometimes questions are vaugue or can be interperted in a lot of different ways when put in the context of a ArduPilot Log. 
<CRITICAL>
    - If you think the question is vaugue or can be interperted in a lot of different ways, ask clarifying questions before proceeding with analysis.
</CRITICAL>

Step 1: Select all relevant message types from the log file using the select_message_types_subagent.
- select_message_types_tool:
    - Input:
        - question: str
            - The user question/request along with any relevant context from previous user questions and answers.

Step 2: Generate the Pandas Code.  
- generate_pandas_code_tool:
    - Input:
        - question: str
            - The user question/request along with any relevant context from previous user questions and answers.
        - message_types: str
            - A list of message types provided by the 'select_message_types_subagent'. 
              Do not attempt to select message types manually.
    - Output:
        - Pandas code that analyzes the data to fulfill the data request.

Step 3: Execute the Pandas query: 
- pandas_executor_tool:  
    - Input: (pandas_code: str, message_types: str) 
        - The pandas_code generated by the generate_pandas_code_tool
        - The message_types should be the same list passed to the generate_pandas_code
    - Output: A pandas dataframe will be generated which has the answer
- If the pandas code fails to execute correct it and pass it into this tool again

Step 3b(Optional): Correct the Pandas code if it fails to execute
- pandas_code_correction_tool:  
    - Input:
        - pandas_code: str
            - The pandas code that failed to execute
        - error_message: str
            - The error message from the failed execution
    - Output:
        - corrected_pandas_code: str
            - The corrected pandas code that should execute without errors
- Do not use this tool unless pandas_executor_tool has failed to execute the code.
- Once you have the corrected pandas code, pass it into the pandas_executor_tool again with the same message_types

Step 4: Outputs the results
- summarize_results_tool:  
    - Output: Will output some of the results from the generated pandas dataframe

Step 5: Give the user the results:
- Summarize the results in a way that is easy to understand for the user. Always give context for the user like where did this data come from and what does it mean.

- Do not use this tool unless pandas_executor_tool has correctly executed
"""


@tool(
    description = """
    This tool will generate a potential list of message types that are relevant to answering the user question.

    - Input:
        - question: str
            - The user question/request along with any relevant context from previous user questions and answers.

    - Output:
        - Pandas code that analyzes the data to fulfill the data request.
    """
)
def select_message_types_tool(user_question: str, state: Annotated[dict, InjectedState]) -> list[str]:

    message_types = "\n".join(sorted(state["message_dfs"].keys()))

    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature = 0.2)

    prompt = f""" 
You are analyzing an ArduPilot .bin log file. The log contains the following message types:
{message_types}

This is the user question that should be answered using the log data:
{user_question}

Consider the following when selecting messages:

* Sensor Lifecycle: Sensor messages (e.g., GPS, IMU, MAG, BARO, BAT, VIBE) may report zero, empty, or sentinel values during initialization, degradation, or failure. Only select messages that reliably record valid values for the relevant parameter. Sensor readings can drift or be incomplete during startup or recovery phases; plan selection logic accordingly.
* Vehicle Lifecycle: The vehicle passes through preflight, arming/takeoff, active flight, landing, and disarmed/shutdown phases. Prefer messages that provide meaningful information across the relevant phase(s). Avoid messages that exist only briefly unless the question specifically concerns that event. Values that seem anomalous or incomplete during preflight or post-flight are normal and should not lead to message exclusion unless relevant to the question.


Note: ArduPilot message types can be broadly categorized as follows:
- Sensors (e.g., GPS, IMU, MAG, BARO, BAT, VIBE) – record physical measurements.
- Flight/Attitude/Navigation (e.g., ATT, CTUN, POS, RATE, XKF*) – record vehicle motion and orientation.
- Commands, Modes, and Control (e.g., CMD, MODE, RCIN, PID*) – record vehicle state, commands sent, and control outputs.
- System info / Parameters / Metadata (e.g., PARM, VER, FMT) – record configuration, system state, and overall flight metadata.
- Errors / Warnings / Status messages (e.g., ERR, MSG) – record faults, warnings, or other important events.

Additionally, the following messages generally provide information throughout the flight and are useful for overall flight summary or timing:  
["ERR", "MSG", "MODE", "HEARTBEAT", "PARM", "VER", "FMT", "STAT", "STATS", "CMD", "MISSION"]

Guidelines for selecting relevant message types:

1. Consider Sensor and Vehicle Lifecycles:
   - Sensor Lifecycle: Sensor messages (e.g., GPS, IMU, MAG, BARO, BAT, VIBE) may report zero, empty, or sentinel values during initialization, degradation, or failure. Only select messages that reliably record valid values for the relevant parameter. Sensor readings can drift or be incomplete during startup or recovery phases; plan selection logic accordingly.
   - Vehicle Lifecycle: The vehicle passes through preflight, arming/takeoff, active flight, landing, and disarmed/shutdown phases. Prefer messages that provide meaningful information across the relevant phase(s). Avoid messages that exist only briefly unless the question specifically concerns that event. Values that seem anomalous or incomplete during preflight or post-flight are normal and should not lead to message exclusion unless relevant to the question.

2. If the question is about a specific flight parameter or sensor:
   - Select the message type(s) that record that parameter.
   - Consider whether the field is reliably populated and not just a placeholder (e.g., temperature may be 0 if no sensor).

3. If the question involves multiple parameters or subsystems:
   - Select all message types that contain relevant data.

4. If the question is about overall flight performance, summary metrics, or timing:
   - Select messages that cover the full flight, record vehicle state, arming/disarming, position, or timestamps.
   - Prefer messages from the list ["ERR", "MSG", "MODE", "HEARTBEAT", "PARM", "VER", "FMT", "STAT", "STATS", "CMD", "MISSION"] if they provide relevant information.
   - Avoid messages that exist only during specific events or short periods 

5. Always think in terms of ArduPilot telemetry and what each message type represents.
   - Do not select messages arbitrarily; select only those necessary to answer the question reliably.

Output format:
- Return only a Python list of message type names.
- Examples:
  - Single message type: ["GPS"]
  - Multiple message types: ["GPS", "ATT", "CTUN"]
- Do not include any extra text, explanation, or punctuation outside the list.

"""

    response = llm.invoke(prompt).content

    return response



@tool(
    description = """
    This tool generates Pandas code to answers a user question using the provided message types.

    - Input:
        - question: str
            - The user question/request along with any relevant context from previous user questions and answers.
        - message_types: str
            - A list of message types provided by the 'select_message_types_subagent'. 
              Do not attempt to select message types manually.
    - Output:
        - Pandas code that analyzes the data to fulfill the data request.
    """
)
def generate_pandas_code_tool(question: str, message_types: list[str], state: Annotated[dict, InjectedState]) -> str:

    dfs_information = """"""

    for msg_type in message_types: 
        if msg_type in state["message_dfs"]:  
            df = state["message_dfs"][msg_type]
            dfs_information += f"{msg_type}\n"
            dfs_information += f"  {getattr(message_information, msg_type)}\n"
            dfs_information += f"  dtypes: {df.dtypes.astype(str).to_dict()}\n"
            if len(df) > 0:
                sample = df.head(1).to_csv(index=False).strip()
            else:
                sample = "No data available in this dataframe"
            
            dfs_information += f"  sample_data:\n{sample}\n"
            dfs_information += "-" * 40 + "\n"  # separator lin

    planner_prompt = f""" 
You are an expert Python data analyst specializing in ArduPilot telemetry. Your goal is to create a step-by-step plan to answer a data request using provided Pandas DataFrame schemas. This plan will be used by another process to generate executable Python code.

---

### **Inputs**

1.  **Data Request:** 
{question}

2.  **Available DataFrames:** 
{dfs_information}

---

### **Critical Rules for Plan Generation**

Your plan **must** create Python steps that adhere to the following strict rules. Any deviation will cause the final code to fail.

* **Execution Environment:** The final code will be run in a restricted scope (`exec()`). Only `pandas` (as `pd`), `numpy` (as `np`), and the initial DataFrames are available.
* **No `lambda` Functions:** `lambda` functions are strictly forbidden.
* **Conditional Logic:** All conditional column creation **must** use vectorized operations with `np.where`.
* **Function Definitions:** Helper functions are allowed but **must** be defined at the top level (`def function_name(...):`). Nested functions are forbidden.
* **Library Usage:** Use **only** `pandas` and `numpy`. Do not import any other libraries.
* **Data Integrity:** Do not create sample DataFrames (`pd.DataFrame(...)`) or hardcode values observed from the data. The plan must be generalizable.

---

### **Your Final Output: The Step-by-Step Plan**

Your response must be a plan containing exactly the following numbered sections.

**0. Understanding the Data**
* **Vehicle Lifecycle Awareness**
  - **Preflight / Initialization:** Vehicle is powered on but not yet armed. Flight modes may be default or undefined (`STABILIZE`, `MANUAL`, etc.). Sensors may report zeros, empty fields, or sentinel values. Telemetry at this stage does **not** indicate active flight.
  - **Arming / Takeoff:** Vehicle transitions from disarmed to armed. The first active flight mode (`AUTO`, `GUIDED`, `LOITER`) indicates the start of actual flight. Sensor data becomes reliable; altitude, attitude, and GPS start reflecting real-world measurements.
  - **Active Flight:** Vehicle is airborne and executing planned maneuvers or mission legs. Telemetry should report valid, stable values for GPS, BARO, ATT, battery, and other sensors. Phase-specific analyses (e.g., mission segments, hover, loiter) can be derived from mode and mission messages.
  - **Landing / Descent:** Vehicle transitions to landing-related modes (`LAND`, `RTL`, `LOITER`). Altitude decreases; speed and thrust may change as motors slow. Telemetry may show temporary anomalies as sensors adjust to changing motion and environment.
  - **Disarmed / Shutdown:** Vehicle is on the ground and motors are off; vehicle is disarmed. Telemetry messages may continue briefly but do **not** indicate active flight. Last recorded flight modes and mission completions should be used to mark flight end.
  - **Telemetry Interpretation Notes:** Always consider **vehicle lifecycle** when analyzing sensor data. Values that seem anomalous during preflight or post-flight may be normal. Plan analyses and filters around lifecycle phases to avoid misinterpreting initialization, takeoff, or landing readings.

* **Telemetry Lifecycle Awareness:** Consider the phases of operation for each sensor and other message types:  
  - **Initialization / Startup:** Field values may be zero, empty, or sentinel defaults. Do not treat these as failures.  
  - **Normal Operation:** Fields report valid, stable values.  
  - **Degradation / Signal Loss:** Field readings may drift, drop, or spike before a fault. Only consider a sensor “lost” if it was previously initialized and valid.  
  - **Failure / Recovery:** Field values may become anomalous or return to normal; plan must account for transitions over time.
* **Telemetry Lifecycle:** Consider the different phases of operation (e.g., initialization, normal flight, failure, recovery). Field values might be zero or nonsensical during initialization; this is not necessarily an error. Plan your logic to account for these different states.
* **Sensor Data:** Define thresholds and conditions for each sensor, and apply them in a way that respects the lifecycle and initialization logic.

**1. Data Selection**
* Identify the primary DataFrames required to answer the request.
* Briefly state the purpose of each selected DataFrame.
* If the same type of data (e.g., altitude, attitude) is present in multiple DataFrames (like `GPS`, `BARO`, `AHR2`), include all relevant sources to allow for comparison or fusion.

**2. Data Filtering**
* Detail the filtering logic needed to isolate the relevant data.
* Example: "Filter the `GPS` DataFrame to only include rows where `Status` is 3 or greater (indicating a 3D fix)."

**3. Feature Engineering**
* Describe any new columns that must be created for the analysis.
* Specify the calculation or transformation logic for each new column.
* Example: "From the `ATT` DataFrame, create a `Roll_Deg` column by converting `Roll` from radians to degrees (`ATT['Roll'] * 180 / np.pi`)."

**4. Grouping and Aggregation**
* Specify which columns to group the data by, if any.
* Define the aggregation functions to apply to the grouped data (e.g., `mean`, `max`, `sum`, `count`).
* Example: "Group the data by the `CTUN.Mode` column and calculate the `mean()` of the `CTUN.Alt` for each mode."

**5. Sorting**
* Describe how the final result should be sorted to present the answer logically.
* Example: "Sort the resulting DataFrame by the `Average_Altitude` column in descending order."

**6. Final Output Structure**
* Describe the structure of the final DataFrame, which **must be stored in a variable named `result_df`**.
* List the exact column names and describe the data each column will contain.
* Example: "The `result_df` will contain two columns: `Flight_Mode` and `Max_Altitude`, where each row represents a unique flight mode and its corresponding maximum altitude."

#WARNINGS:

* Do NOT create or populate random DataFrames. Use only the DataFrames provided. 
* Do not attempt to create new DataFrames with pd.DataFrame() to simulate missing data
* Do NOT import or use any libraries beyond pandas (pd) and numpy (np).
* Avoid lambda functions and nested functions. Only top-level helper functions are allowed.
* Do NOT hardcode values from inspection of the data; use the actual DataFrame content.
* Respect the execution scope (exec() in ldict) — do not rely on variables outside of ldict.
* Always handle missing or anomalous data using pandas/numpy; do not assume the data is perfect.

"""

    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature = 0.2)

    pandas_plan = llm.invoke(planner_prompt).content

    pandas_generator_prompt = f""" 
# ROLE: Expert Python Writer

You will be given the following data request to fufill:
{question}

The following dataframes to minipulate:
{dfs_information}

Python Plan Created to answer the question
{pandas_plan}

---
Your task is to write a complete, executable python code that follows the python plan.

#WARNINGS:
* Do NOT create or populate random DataFrames. Use only the DataFrames provided. 
* Do not attempt to create new DataFrames with pd.DataFrame() to simulate missing data
* Do NOT import or use any libraries beyond pandas (pd) and numpy (np).
* Avoid lambda functions and nested functions. Only top-level helper functions are allowed.
* Do NOT hardcode values from inspection of the data; use the actual DataFrame content.
* Respect the execution scope (exec() in ldict) — do not rely on variables outside of ldict.
* Always handle missing or anomalous data using pandas/numpy; do not assume the data is perfect.


# CRITICAL OUTPUT FORMAT REQUIREMENTS:
- Python Code Only: Your response must consist solely of the Python code. Do not include any accompanying text, explanations, or comments (unless absolutely essential for inline code clarity). Do not include any markdown, non python formatting. The returned code should be directly executable in if put into exec(). Which means only python code is allowed.
   **ABSOLUTELY NO MARKDOWN FORMATTING ALLOWED**
   **DO NOT USE ```python or ``` or any code blocks**
   **DO NOT USE ANY BACKTICKS**
   **DO NOT USE ANY MARKDOWN SYNTAX**
   **Your response must be ONLY raw Python code that can be directly executed with exec().**
- DataFrame Name: Assume the data is already loaded into a pandas DataFrames
- Result Variable: The final output of your analysis or manipulation should be stored in a pandas DataFrame named result_df. If the result is a Series, a scalar, or another type, convert it to a DataFrame named result_df if appropriate, or store it in result_df directly if it's already a DataFrame.
- Import Pandas: Include the import statement for the Pandas library at the beginning of your code.
- Do not include any comments in the code.

When writing code remember the following:
Always close brackets, parentheses, and quotes.

REMEMBER: Your entire response should be executable Python code without any formatting or explanations.
"""
    

    pandas_code = llm.with_structured_output(
        StructuredPandasQueryOutput, include_raw = True
    ).invoke(pandas_generator_prompt).get("parsed").to_dict()["pandas_code"]

    return pandas_code 


@tool(
    description = "Given pandas code and a list of messages types to minipulate will execute the pandas code and store the result in a dataframe "
)
def pandas_executor_tool(pandas_code:str, message_types: list[str], state: Annotated[dict, InjectedState], tool_call_id: Annotated[str, InjectedToolCallId] ):
    ldict = { "pd": pd, "np": np,}

    for msg_type in message_types: 
        if msg_type in message_dfs:  
            ldict[msg_type] = message_dfs[msg_type]

    try:
        # Execute user-provided code
        exec(pandas_code, globals(), ldict)

        # Retrieve the DataFrame from local dict
        global result_df
        result_df = ldict.get("result_df", None)

        if result_df is None:
            return "result_df was not generated"
        else:
            return Command(update={
                "result_df": result_df,
                "messages": [ToolMessage("Pandas Code Successfully Executed", tool_call_id=tool_call_id)]
    })

    except Exception:
        # Log the exception
        error_message = traceback.format_exc()
        return f"Execution error: {error_message}"
        


@tool(
    description="""A tool that allows you to correct pandas code that has failed to execute. 
    
    Required Input:
    - pandas_code: str
        - The pandas code that failed to execute
    - error_message: str
        - The error message from the failed execution
    
    Output:
    - corrected_pandas_code: str
        - The corrected pandas code that should execute without errors
    """
)
def pandas_code_correction_tool(pandas_code: str, error_message: str) -> str:
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature = 0.2)

    correction_prompt = f""" 
You are an expert Python programmer. You will be given a piece of pandas code that has failed to execute along with the error message from the failed execution. Your task is to correct the pandas code so that it can execute without errors.

**EXECUTION ENVIRONMENT - CRITICAL:**
Your generated code will be executed in a secure, isolated environment using Python's exec() function. A scope is prepared for exec() where the following objects are made directly available as variables:
    - df: A pandas DataFrame containing the data.
    - pd: The pandas module.
    - np: The numpy module.
    - json: The json module.

- Since the code will be executed in a remote code environment only the following python modules are available: pandas as pd, numpy as np, json as json
- Do not attempt to install/use modules unavailable in the remote code environment.

**CRITICAL SCOPE & CODING CONSTRAINTS - MUST FOLLOW:**

* **No `lambda` functions:** NEVER use `lambda` functions. They cannot access variables and helper functions defined in the limited `exec()` scope, which will cause a `NameError`.
* **Use Vectorized Conditionals:** For conditional logic (like creating a new column based on values in another), you **MUST** use `np.where`. This is the only reliable method in the target environment. For example:
    * **Incorrect (will fail):** `df['new_col'] = df['old_col'].apply(lambda x: 'A' if x in my_list else 'B')`
    * **Correct (use this pattern):** `df['new_col'] = np.where(df['old_col'].isin(my_list), 'A', 'B')`
* **Top-Level Functions Only:** All `def` statements for helper functions must be at the top level of the code block. Do not use nested functions or closures.
* **Access via `ldict`:** All variables and functions are only accessible through the `ldict` dictionary.
* **List Comprehensions for Transformations:** Use list comprehensions for simple transformations if `np.where` is not applicable.
* **USE ONLY THE MODULES THAT THE SECURE REMOTE CODE ENVIRONMENT HAS ACCESS TO: pandas as pd, numpy as np, json as json.
* DO not attempt to install/use modules unavailable in the remote code environment.

This is the pandas code that failed to execute:
{pandas_code}

This is the error message from the failed execution:
{error_message}

**REQUIREMENTS:**
- Fix the error in the provided Python code, adhering strictly to the scope constraints above.
- Do NOT create new sample data; operate on the provided `df`.
- Return ONLY the corrected Python code as plain text.
- Do not include any comments in the code.

**ABSOLUTELY NO MARKDOWN FORMATTING ALLOWED**
**DO NOT USE ```python or ``` or any code blocks**
**DO NOT USE ANY BACKTICKS**
**Your response must be ONLY raw Corrected Python code that can be directly executed with exec().**
"""
    
    corrected_code = llm.invoke(correction_prompt).content

    return corrected_code


@tool(
    description="Outputs some of the rows from the result_df "
)
def summarize_results_tool():
    try:
        temp = result_df.head(5).to_string(index=False)
        return temp
    except Exception as e:
        return f"Error generating summary: {str(e)}"


async def create_graph():

    agent = create_deep_agent(
        tools=[
            generate_pandas_code_tool,
            pandas_executor_tool,
            summarize_results_tool,
            select_message_types_tool,
            pandas_code_correction_tool
        ],
        builtin_tools=["write_todos"],
        instructions=system_prompt,
        model=ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.2),
        subagents=[
        ],
        state_schema=AgentState,
    ).with_config({"recursion_limit": 100})

    return agent

agent = create_graph()


message_dfs = bin_to_dataframe_optimized("1980-01-08 09-44-08.bin")


state = {
    "result_df": "",
    "message_dfs": message_dfs,
    "messages": [{"role": "user", "content": "what is the weather in sf"}]
}


agent.invoke(state)

In [ ]:
message_dfs = bin_to_dataframe_optimized("1980-01-08 09-44-08.bin")

In [11]:
state = {
    "result_df": "",
    "message_dfs": message_dfs,
    "messages": [{"role": "user", "content": "What was the highest altitude reached during the flight?"}]
}

In [12]:
state = {
    "result_df": "",
    "message_dfs": message_dfs,
    "messages": [{"role": "user", "content": "What was the highest altitude reached during the flight?"}]
}

agent = await create_graph()
result = agent.invoke(state)

E0000 00:00:1758263271.102859  285434 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1758263329.895145  297380 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1758263409.919252  298412 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [16]:
result['messages'][-1].content

"The highest altitude reached during the flight was approximately 124.26 meters. This is based on the 'Alt' column in the POS (Position) data."

In [ ]:
import time 
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature = 0.2)

start = time.time()
test = llm.invoke("explain fast api to me in detail as much deatil as possible").content
print(time.time() - start)

E0000 00:00:1758295138.686859  285434 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
